# GeoAI – Geolocation Estimation from Street View Images

Dieses Notebook trainiert ein CNN-Modell (Transfer Learning mit EfficientNet/ResNet),
das anhand eines Fotos vorhersagt, **in welchem Land und an welchem genauen Ort** das Bild aufgenommen wurde.

**Ansatz (basierend auf ECCV 2018 Paper):**
1. Welt in geografische Gitterzellen einteilen
2. CNN klassifiziert jedes Bild einer Zelle zu
3. Vorhersage → Lat/Lon (Zellmittelpunkt) → Reverse Geocoding → Land + Adresse
4. Ergebnis vs. Ground Truth vergleichen

**Dataset:** [paulchambaz/google-street-view](https://www.kaggle.com/datasets/paulchambaz/google-street-view)

## 1. Setup & Installation

In [ ]:
# Alle benötigten Pakete installieren
import subprocess, sys

packages = [
    'kagglehub',
    'torch', 'torchvision',
    'timm',              # EfficientNet / moderne Backbone-Modelle
    'geopy',             # Reverse Geocoding (Nominatim, kostenlos)
    'folium',            # interaktive Karte
    'Pillow',
    'pandas', 'numpy',
    'matplotlib', 'seaborn',
    'tqdm',
    'scikit-learn',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('✅ Alle Pakete installiert/vorhanden')

In [ ]:
import os
import math
import json
import time
import random
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import timm
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from sklearn.model_selection import train_test_split

# Reproduzierbarkeit
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Gerät: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## 2. Dataset herunterladen

In [ ]:
import kagglehub

# Dataset herunterladen (cached nach erstem Download)
DATASET_PATH = kagglehub.dataset_download('paulchambaz/google-street-view')
DATASET_PATH = Path(DATASET_PATH)

print(f'📁 Dataset Pfad: {DATASET_PATH}')
print('\nDateistruktur:')
for f in sorted(DATASET_PATH.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        print(f'  {f.relative_to(DATASET_PATH)} ({size_mb:.1f} MB)')

## 3. Dataset-Struktur erkunden & Metadaten laden

In [ ]:
def find_metadata_file(base: Path) -> Optional[Path]:
    """Sucht die CSV/JSON Metadaten-Datei im Dataset."""
    for ext in ['*.csv', '*.json', '*.txt', '*.tsv']:
        files = list(base.rglob(ext))
        if files:
            return files[0]
    return None

def find_image_dirs(base: Path):
    """Findet alle Ordner mit Bilddateien."""
    img_exts = {'.jpg', '.jpeg', '.png', '.webp'}
    dirs = set()
    for f in base.rglob('*'):
        if f.suffix.lower() in img_exts:
            dirs.add(f.parent)
    return list(dirs)

META_FILE = find_metadata_file(DATASET_PATH)
IMG_DIRS  = find_image_dirs(DATASET_PATH)

print(f'Metadaten-Datei : {META_FILE}')
print(f'Bild-Ordner     : {IMG_DIRS[:5]}')

In [ ]:
# ──────────────────────────────────────────────
#  Metadaten einlesen & Spalten normalisieren
# ──────────────────────────────────────────────

def load_metadata(meta_file: Path) -> pd.DataFrame:
    if meta_file.suffix == '.csv':
        df = pd.read_csv(meta_file)
    elif meta_file.suffix == '.json':
        df = pd.read_json(meta_file)
    elif meta_file.suffix in ('.txt', '.tsv'):
        df = pd.read_csv(meta_file, sep='\t')
    else:
        raise ValueError(f'Unbekanntes Format: {meta_file.suffix}')
    return df

# Spalten-Aliase für verschiedene Dataset-Benennungen
LAT_ALIASES = ['lat', 'latitude', 'Lat', 'Latitude', 'LAT', 'LATITUDE']
LON_ALIASES = ['lon', 'lng', 'longitude', 'Long', 'Longitude', 'LON', 'LONGITUDE', 'LNG']
IMG_ALIASES = ['filename', 'image', 'img', 'file', 'path', 'image_id', 'id',
               'Filename', 'Image', 'File', 'img_path', 'image_path']

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Benennt Spalten einheitlich um: lat, lon, filename."""
    rename = {}
    for col in df.columns:
        if col in LAT_ALIASES:
            rename[col] = 'lat'
        elif col in LON_ALIASES:
            rename[col] = 'lon'
        elif col in IMG_ALIASES and 'filename' not in rename.values():
            rename[col] = 'filename'
    return df.rename(columns=rename)

if META_FILE:
    df_meta = load_metadata(META_FILE)
    df_meta = normalize_columns(df_meta)
    print('Spalten:', df_meta.columns.tolist())
    print(f'Anzahl Einträge: {len(df_meta):,}')
    df_meta.head(3)

In [ ]:
# Wenn kein Metadaten-CSV: Bilder scannen & Koordinaten aus Dateinamen extrahieren
# (manche Datasets kodieren lat_lon im Dateinamen, z.B. "48.8566_2.3522.jpg")

def scan_images_as_metadata(img_dirs) -> pd.DataFrame:
    rows = []
    img_exts = {'.jpg', '.jpeg', '.png', '.webp'}
    for d in img_dirs:
        for f in d.iterdir():
            if f.suffix.lower() in img_exts:
                row = {'filename': str(f), 'lat': np.nan, 'lon': np.nan}
                # Versuche lat/lon aus Dateiname zu parsen (Format: lat_lon.jpg)
                parts = f.stem.replace(',', '_').replace(' ', '_').split('_')
                nums = []
                for p in parts:
                    try:
                        nums.append(float(p))
                    except ValueError:
                        pass
                if len(nums) >= 2:
                    row['lat'], row['lon'] = nums[0], nums[1]
                rows.append(row)
    return pd.DataFrame(rows)

if META_FILE is None:
    print('Kein Metadaten-CSV gefunden – scanne Bilder...')
    df_meta = scan_images_as_metadata(IMG_DIRS)
    print(f'{len(df_meta):,} Bilder gefunden')

# Sicherstellen dass lat/lon existieren
assert 'lat' in df_meta.columns and 'lon' in df_meta.columns, \
    f'Spalten lat/lon nicht gefunden. Vorhandene Spalten: {df_meta.columns.tolist()}'

# Ungültige Koordinaten entfernen
df_meta = df_meta.dropna(subset=['lat', 'lon'])
df_meta = df_meta[(df_meta['lat'].between(-90, 90)) & (df_meta['lon'].between(-180, 180))]
df_meta = df_meta.reset_index(drop=True)

print(f'\n✅ Gültige Einträge mit Koordinaten: {len(df_meta):,}')

# Bild-Pfad-Spalte aufbauen (falls nur Dateiname vorhanden)
if 'filename' in df_meta.columns:
    def resolve_path(fn):
        p = Path(fn)
        if p.exists():
            return str(p)
        # Suche im Dataset-Verzeichnis
        candidates = list(DATASET_PATH.rglob(p.name))
        return str(candidates[0]) if candidates else str(fn)
    df_meta['img_path'] = df_meta['filename'].apply(resolve_path)
else:
    df_meta['img_path'] = df_meta['filename']

df_meta[['img_path', 'lat', 'lon']].head(3)

## 4. Geografische Gitterzellen erstellen

Die Welt wird in ein regelmäßiges Gitter eingeteilt. Jede Zelle ist eine Klasse für den Klassifikator.
Zellen ohne Trainingsbilder werden ignoriert.

In [ ]:
# ──────────────────────────────────────────────
#  Konfiguration
# ──────────────────────────────────────────────
CFG = {
    # Gittergröße in Grad (kleiner = präziser, aber mehr Klassen nötig)
    'grid_deg'       : 5.0,    # 5°×5° Zellen (~550 km)
    'min_imgs_per_cell': 5,    # Zellen mit weniger Bildern werden verworfen

    # Modell
    'backbone'       : 'efficientnet_b4',   # timm-Modellname
    'img_size'       : 224,
    'dropout'        : 0.3,

    # Training
    'batch_size'     : 32,
    'epochs'         : 15,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-4,
    'val_split'      : 0.15,
    'test_split'     : 0.10,

    # Pfade
    'checkpoint_dir' : 'checkpoints',
    'best_model'     : 'checkpoints/best_model.pth',
}

Path(CFG['checkpoint_dir']).mkdir(exist_ok=True)

# ──────────────────────────────────────────────
#  Zellen-ID aus Koordinaten berechnen
# ──────────────────────────────────────────────
STEP = CFG['grid_deg']

def coords_to_cell(lat: float, lon: float) -> str:
    """Gibt Zellen-ID als String zurück, z.B. 'lat50_lon10'."""
    lat_bin = math.floor(lat / STEP) * STEP
    lon_bin = math.floor(lon / STEP) * STEP
    return f'{lat_bin:.1f}_{lon_bin:.1f}'

def cell_center(cell_id: str):
    """Gibt (lat, lon) des Zellmittelpunkts zurück."""
    lat_bin, lon_bin = map(float, cell_id.split('_'))
    return lat_bin + STEP / 2, lon_bin + STEP / 2

# Zelle für jedes Bild bestimmen
df_meta['cell_id'] = df_meta.apply(
    lambda r: coords_to_cell(r['lat'], r['lon']), axis=1
)

# Seltene Zellen entfernen
cell_counts = df_meta['cell_id'].value_counts()
valid_cells = cell_counts[cell_counts >= CFG['min_imgs_per_cell']].index
df_meta = df_meta[df_meta['cell_id'].isin(valid_cells)].reset_index(drop=True)

# Klassen-Index
unique_cells = sorted(df_meta['cell_id'].unique())
cell_to_idx  = {c: i for i, c in enumerate(unique_cells)}
idx_to_cell  = {i: c for c, i in cell_to_idx.items()}
NUM_CLASSES  = len(unique_cells)

df_meta['label'] = df_meta['cell_id'].map(cell_to_idx)

print(f'Anzahl Zellen (Klassen) : {NUM_CLASSES}')
print(f'Bilder nach Filterung   : {len(df_meta):,}')

# Speichere Zellen-Mapping
with open('checkpoints/cell_mapping.json', 'w') as f:
    json.dump({'cell_to_idx': cell_to_idx, 'idx_to_cell': idx_to_cell, 'step': STEP}, f)

In [ ]:
# ── Verteilung visualisieren ──
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Weltkartenplot
ax = axes[0]
ax.set_facecolor('#d4e9f7')
scatter = ax.scatter(df_meta['lon'], df_meta['lat'],
                     c=df_meta['label'], cmap='tab20',
                     s=1, alpha=0.4)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title(f'Geografische Verteilung ({len(df_meta):,} Bilder, {NUM_CLASSES} Zellen)')
ax.set_xlabel('Längengrad'); ax.set_ylabel('Breitengrad')
ax.grid(True, alpha=0.3)

# Top-20 Zellen nach Bildanzahl
axes[1].barh(range(20), cell_counts[:20].values, color='steelblue')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(cell_counts[:20].index, fontsize=7)
axes[1].set_xlabel('Anzahl Bilder')
axes[1].set_title('Top-20 Zellen nach Bildanzahl')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Train / Val / Test Split

In [ ]:
# Stratifizierter Split (jede Zelle in allen Sets vertreten)
df_trainval, df_test = train_test_split(
    df_meta, test_size=CFG['test_split'],
    stratify=df_meta['label'], random_state=SEED
)
df_train, df_val = train_test_split(
    df_trainval,
    test_size=CFG['val_split'] / (1 - CFG['test_split']),
    stratify=df_trainval['label'], random_state=SEED
)

print(f'Train : {len(df_train):>6,}  ({100*len(df_train)/len(df_meta):.0f}%)')
print(f'Val   : {len(df_val):>6,}  ({100*len(df_val)/len(df_meta):.0f}%)')
print(f'Test  : {len(df_test):>6,}  ({100*len(df_test)/len(df_meta):.0f}%)')

## 6. PyTorch Dataset & DataLoader

In [ ]:
IMG_MEAN = [0.485, 0.456, 0.406]  # ImageNet Normalisierung
IMG_STD  = [0.229, 0.224, 0.225]
SZ = CFG['img_size']

train_tf = transforms.Compose([
    transforms.Resize((SZ + 32, SZ + 32)),
    transforms.RandomCrop(SZ),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

class GeoDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['label'])
        # Echte Koordinaten für spätere Evaluation mitgeben
        lat, lon = float(row['lat']), float(row['lon'])
        return img, label, lat, lon

train_ds  = GeoDataset(df_train, train_tf)
val_ds    = GeoDataset(df_val,   val_tf)
test_ds   = GeoDataset(df_test,  val_tf)

NUM_WORKERS = 4 if os.name != 'nt' else 0   # Windows: 0 Workers

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train-Batches : {len(train_loader)}')
print(f'Val-Batches   : {len(val_loader)}')

# Beispiel-Bilder anzeigen
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    row = df_train.iloc[i]
    try:
        img = Image.open(row['img_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'lat={row["lat"]:.2f}, lon={row["lon"]:.2f}', fontsize=7)
    except Exception:
        ax.text(0.5, 0.5, 'Ladefehler', ha='center')
    ax.axis('off')
plt.suptitle('Beispiel-Trainingsbilder', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Modell-Architektur (EfficientNet + Geo-Klassifikator)

In [ ]:
class GeoModel(nn.Module):
    """Pretrained CNN → geografische Zellen-Klassifikation."""

    def __init__(self, backbone_name: str, num_classes: int, dropout: float = 0.3):
        super().__init__()
        # Backbone laden (pretrained auf ImageNet)
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0  # num_classes=0 → nur Features
        )
        feat_dim = self.backbone.num_features

        # Klassifikationskopf
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)   # (B, feat_dim)
        return self.head(features)    # (B, num_classes)


model = GeoModel(CFG['backbone'], NUM_CLASSES, CFG['dropout']).to(DEVICE)

total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Backbone      : {CFG["backbone"]}')
print(f'Klassen       : {NUM_CLASSES}')
print(f'Parameter     : {total_params:,}  (trainierbar: {train_params:,})')

## 8. Distanz-Metriken

Wie im ECCV-Paper: Haversine-Distanz zwischen vorhergesagtem und echtem Standort.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    """Großkreis-Distanz in km zwischen zwei GPS-Punkten."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def batch_haversine(pred_cells, true_lats, true_lons, idx_to_cell):
    """Berechnet Haversine-Distanz für einen Batch."""
    dists = []
    for pred_idx, t_lat, t_lon in zip(pred_cells, true_lats, true_lons):
        p_lat, p_lon = cell_center(idx_to_cell[pred_idx])
        dists.append(haversine_km(p_lat, p_lon, t_lat, t_lon))
    return dists

# Präzisionsthresholds wie im Paper
THRESHOLDS = {
    'Straße'     :    1,    # 1 km
    'Stadt'      :   25,
    'Region'     :  200,
    'Land'       :  750,
    'Kontinent'  : 2500,
}

print('Distanz-Schwellwerte:', THRESHOLDS)

## 9. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'],
                         weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6
)

# ──────────────────────────────────────────────
def run_epoch(loader, mode='train'):
    is_train = mode == 'train'
    model.train(is_train)
    total_loss, total_correct, total_dist = 0.0, 0, []

    with torch.set_grad_enabled(is_train):
        for imgs, labels, t_lats, t_lons in tqdm(loader, desc=mode, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_loss    += loss.item() * len(imgs)

            # Haversine-Distanz
            dist = batch_haversine(
                preds.cpu().numpy(),
                t_lats.numpy(), t_lons.numpy(),
                idx_to_cell
            )
            total_dist.extend(dist)

    n  = len(loader.dataset)
    return {
        'loss'      : total_loss / n,
        'acc'       : total_correct / n,
        'median_km' : np.median(total_dist),
        'mean_km'   : np.mean(total_dist),
    }

# ──────────────────────────────────────────────
history = []
best_val_loss = float('inf')

for epoch in range(1, CFG['epochs'] + 1):
    t_start = time.time()
    train_m = run_epoch(train_loader, 'train')
    val_m   = run_epoch(val_loader,   'val')
    scheduler.step()

    elapsed = time.time() - t_start
    lr_now  = optimizer.param_groups[0]['lr']

    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_m.items()},
                            **{f'val_{k}': v   for k, v in val_m.items()},
                            'lr': lr_now}
    history.append(row)

    # Checkpoint bei Verbesserung
    if val_m['loss'] < best_val_loss:
        best_val_loss = val_m['loss']
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optimizer'   : optimizer.state_dict(),
            'val_loss'    : best_val_loss,
            'num_classes' : NUM_CLASSES,
            'cfg'         : CFG,
        }, CFG['best_model'])
        marker = ' ⭐ (best)'
    else:
        marker = ''

    print(f'Epoch {epoch:02d}/{CFG["epochs"]} | '
          f'Loss: {train_m["loss"]:.4f}/{val_m["loss"]:.4f} | '
          f'Acc: {100*train_m["acc"]:.1f}%/{100*val_m["acc"]:.1f}% | '
          f'Median-km: {val_m["median_km"]:.0f} | '
          f'LR: {lr_now:.1e} | {elapsed:.0f}s{marker}')

df_history = pd.DataFrame(history)

In [ ]:
# Trainingskurven
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df_history['epoch'], df_history['train_loss'], label='Train')
axes[0].plot(df_history['epoch'], df_history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(df_history['epoch'], df_history['train_acc'] * 100, label='Train')
axes[1].plot(df_history['epoch'], df_history['val_acc']   * 100, label='Val')
axes[1].set_title('Genauigkeit (%)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(df_history['epoch'], df_history['val_median_km'], color='darkorange')
axes[2].set_title('Median Distanz (km) – Val'); axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('km')

plt.suptitle('Trainingsverlauf', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Evaluation auf Test-Set

In [ ]:
# Bestes Modell laden
ckpt = torch.load(CFG['best_model'], map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'✅ Bestes Modell geladen (Epoch {ckpt["epoch"]}, Val-Loss: {ckpt["val_loss"]:.4f})')

# ── Vollständige Evaluation ──
model.eval()
all_preds, all_labels, all_t_lats, all_t_lons = [], [], [], []
all_dists = []

with torch.no_grad():
    for imgs, labels, t_lats, t_lons in tqdm(test_loader, desc='Test-Eval'):
        logits = model(imgs.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_t_lats.extend(t_lats.numpy())
        all_t_lons.extend(t_lons.numpy())
        all_dists.extend(batch_haversine(preds, t_lats.numpy(), t_lons.numpy(), idx_to_cell))

all_dists = np.array(all_dists)
test_acc  = np.mean(np.array(all_preds) == np.array(all_labels))

print(f'\n📊 Test-Ergebnisse')
print(f'Zell-Genauigkeit  : {100*test_acc:.2f}%')
print(f'Median-Distanz    : {np.median(all_dists):,.0f} km')
print(f'Mittlere Distanz  : {np.mean(all_dists):,.0f} km')
print()
print('Präzision @ Schwellwert:')
for name, km in THRESHOLDS.items():
    pct = 100 * np.mean(all_dists < km)
    print(f'  {name:<12} (<{km:>5} km) : {pct:.1f}%')

In [ ]:
# Distanz-Histogramm
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(all_dists, bins=80, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(all_dists), color='red', linestyle='--',
                label=f'Median: {np.median(all_dists):,.0f} km')
axes[0].set_xlabel('Distanz (km)'); axes[0].set_ylabel('Anzahl Bilder')
axes[0].set_title('Fehler-Verteilung (Test-Set)')
axes[0].legend()

# Kumulative Kurve (wie im Paper)
sorted_d = np.sort(all_dists)
cdf      = np.arange(1, len(sorted_d)+1) / len(sorted_d)
axes[1].plot(sorted_d, cdf * 100)
for name, km in THRESHOLDS.items():
    pct = 100 * np.mean(all_dists < km)
    axes[1].axvline(km, linestyle=':', alpha=0.7)
    axes[1].annotate(f'{name}\n{pct:.0f}%', xy=(km, pct),
                     fontsize=7, color='darkred',
                     xytext=(km + 100, pct - 5))
axes[1].set_xscale('log')
axes[1].set_xlabel('Distanz (km, log-Skala)')
axes[1].set_ylabel('% korrekt innerhalb Distanz')
axes[1].set_title('Kumulative Präzisionskurve (GCD-Kurve)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 11. Reverse Geocoding

Vorhergesagte Koordinaten → **Ortsname** (Land, Stadt, Adresse) via Nominatim

In [ ]:
geolocator = Nominatim(user_agent='geoai_notebook_v1', timeout=10)

def reverse_geocode(lat: float, lon: float, retries: int = 3) -> dict:
    """Konvertiert Koordinaten in Ortsname mit Fallback."""
    for attempt in range(retries):
        try:
            loc = geolocator.reverse((lat, lon), language='de', zoom=10)
            if loc is None:
                return {'display': 'Unbekannt', 'country': '?', 'city': '?'}
            addr = loc.raw.get('address', {})
            city = (addr.get('city') or addr.get('town') or
                    addr.get('village') or addr.get('county') or '?')
            return {
                'display': loc.address,
                'country': addr.get('country', '?'),
                'country_code': addr.get('country_code', '?').upper(),
                'city'   : city,
            }
        except GeocoderTimedOut:
            time.sleep(1)
    return {'display': 'Timeout', 'country': '?', 'city': '?'}

# Testen
test_result = reverse_geocode(48.8566, 2.3522)
print('Test (Paris):', test_result)

## 12. Einzelbild-Vorhersage mit Vergleich Ergebnis vs. Wahrheit

In [ ]:
def predict_image(img_path: str, true_lat: float = None, true_lon: float = None,
                  top_k: int = 3):
    """
    Vorhersage für ein einzelnes Bild.
    Gibt vorhergesagten Ort aus und vergleicht mit Ground Truth.
    """
    img = Image.open(img_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)[0]
        probs  = torch.softmax(logits, dim=0)

    top_probs, top_idxs = probs.topk(top_k)
    top_probs = top_probs.cpu().numpy()
    top_idxs  = top_idxs.cpu().numpy()

    # Top-1 Vorhersage
    best_idx = top_idxs[0]
    pred_lat, pred_lon = cell_center(idx_to_cell[best_idx])
    pred_geo = reverse_geocode(pred_lat, pred_lon)

    # Anzeige
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bild
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Eingabebild', fontsize=12)

    # Karte mit Vorhersage vs. Wahrheit
    axes[1].set_facecolor('#cce5ff')
    axes[1].set_xlim(-180, 180)
    axes[1].set_ylim(-90, 90)
    axes[1].set_xlabel('Längengrad'); axes[1].set_ylabel('Breitengrad')
    axes[1].grid(True, alpha=0.3)

    # Vorhersage (rot)
    axes[1].scatter(pred_lon, pred_lat, c='red', s=200, zorder=5,
                    marker='*', label='Vorhersage')

    # Ground Truth (grün)
    if true_lat is not None and true_lon is not None:
        axes[1].scatter(true_lon, true_lat, c='green', s=150, zorder=5,
                        marker='o', label='Wahrheit')
        dist = haversine_km(pred_lat, pred_lon, true_lat, true_lon)
        true_geo = reverse_geocode(true_lat, true_lon)
    else:
        dist = None
        true_geo = None

    axes[1].legend(fontsize=10)

    # Titel mit Ergebnissen
    info_lines = [
        f'🔴 Vorhersage: {pred_geo["city"]}, {pred_geo["country"]}',
        f'   Koordinaten: {pred_lat:.2f}°N, {pred_lon:.2f}°E',
        f'   Konfidenz  : {100*top_probs[0]:.1f}%',
    ]
    if true_geo:
        info_lines += [
            f'🟢 Wahrheit  : {true_geo["city"]}, {true_geo["country"]}',
            f'   Koordinaten: {true_lat:.2f}°N, {true_lon:.2f}°E',
            f'   Abstand    : {dist:,.0f} km',
        ]
    axes[1].set_title('\n'.join(info_lines), fontsize=9, loc='left')

    plt.tight_layout()
    plt.show()

    # Top-K ausgeben
    print('\nTop-K Vorhersagen:')
    for rank, (idx, prob) in enumerate(zip(top_idxs, top_probs), 1):
        lat_c, lon_c = cell_center(idx_to_cell[idx])
        geo = reverse_geocode(lat_c, lon_c)
        print(f'  {rank}. {geo["city"]}, {geo["country"]} '
              f'({lat_c:.1f}°N, {lon_c:.1f}°E) – {100*prob:.1f}%')

    return {'pred_lat': pred_lat, 'pred_lon': pred_lon,
            'pred_country': pred_geo['country'],
            'pred_city': pred_geo['city'],
            'confidence': float(top_probs[0]),
            'distance_km': dist}


# ── Demo: Zufälliges Testbild ──
sample = df_test.sample(1).iloc[0]
result = predict_image(
    img_path  = sample['img_path'],
    true_lat  = sample['lat'],
    true_lon  = sample['lon'],
)

## 13. Massen-Test: Mehrere Bilder vergleichen

In [ ]:
N_SAMPLES = 16   # Anzahl Testbilder
samples   = df_test.sample(N_SAMPLES, random_state=SEED)

results = []
for _, row in tqdm(samples.iterrows(), total=N_SAMPLES, desc='Vorhersagen'):
    img = Image.open(row['img_path']).convert('RGB')
    t   = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_idx = model(t).argmax(dim=1).item()
    p_lat, p_lon = cell_center(idx_to_cell[pred_idx])
    geo_pred = reverse_geocode(p_lat, p_lon)
    geo_true = reverse_geocode(float(row['lat']), float(row['lon']))
    dist     = haversine_km(p_lat, p_lon, row['lat'], row['lon'])
    results.append({
        'img_path'    : row['img_path'],
        'true_lat'    : row['lat'], 'true_lon' : row['lon'],
        'pred_lat'    : p_lat,      'pred_lon' : p_lon,
        'true_country': geo_true['country'],
        'pred_country': geo_pred['country'],
        'true_city'   : geo_true['city'],
        'pred_city'   : geo_pred['city'],
        'distance_km' : dist,
    })

df_results = pd.DataFrame(results)

# Zusammenfassung
country_acc = (df_results['true_country'] == df_results['pred_country']).mean()
print(f'Länder-Genauigkeit  : {100*country_acc:.1f}%')
print(f'Median-Distanz (km) : {df_results["distance_km"].median():,.0f}')
df_results[['true_country','pred_country','true_city','pred_city','distance_km']].head(N_SAMPLES)

In [ ]:
# Bildgitter mit Ergebnissen
cols = 4
rows_n = math.ceil(N_SAMPLES / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 4, rows_n * 3.5))

for i, (ax, (_, r)) in enumerate(zip(axes.flat, df_results.iterrows())):
    try:
        img = Image.open(r['img_path']).convert('RGB')
        ax.imshow(img)
    except Exception:
        ax.text(0.5, 0.5, 'Fehler', ha='center')

    color = 'green' if r['distance_km'] < 750 else ('orange' if r['distance_km'] < 2500 else 'red')
    ax.set_title(
        f'Wahr : {r["true_city"]}, {r["true_country"]}\n'
        f'Pred : {r["pred_city"]}, {r["pred_country"]}\n'
        f'Dist : {r["distance_km"]:,.0f} km',
        fontsize=6.5, color=color
    )
    ax.axis('off')

for ax in axes.flat[N_SAMPLES:]:
    ax.axis('off')

plt.suptitle('Vorhersage vs. Wahrheit (🟢 < 750 km  🟠 < 2500 km  🔴 > 2500 km)', fontsize=11)
plt.tight_layout()
plt.show()

## 14. Interaktive Karte (Folium)

In [ ]:
import folium

m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

for _, r in df_results.iterrows():
    # Wahres Standort (blau)
    folium.CircleMarker(
        location=[r['true_lat'], r['true_lon']],
        radius=6, color='blue', fill=True, fill_opacity=0.8,
        popup=f"Wahrheit: {r['true_city']}, {r['true_country']}",
    ).add_to(m)

    # Vorhersage (rot)
    folium.CircleMarker(
        location=[r['pred_lat'], r['pred_lon']],
        radius=6, color='red', fill=True, fill_opacity=0.8,
        popup=f"Vorhersage: {r['pred_city']}, {r['pred_country']} ({r['distance_km']:,.0f} km)",
    ).add_to(m)

    # Linie zwischen Vorhersage und Wahrheit
    folium.PolyLine(
        locations=[[r['true_lat'], r['true_lon']], [r['pred_lat'], r['pred_lon']]],
        color='gray', weight=1, opacity=0.6,
    ).add_to(m)

legend_html = '''
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border-radius:5px;font-size:12px;">
  <b>Legende</b><br>
  🔵 Wahre Position<br>🔴 Vorhersage<br>── Abstand
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
m.save('geo_predictions_map.html')
print('✅ Karte gespeichert: geo_predictions_map.html')
m  # Zeige Karte inline im Notebook

## 15. Inference auf beliebiges Bild (Standalone)

Einfach `IMG_TO_PREDICT` anpassen und diese Zelle ausführen – keine erneutes Training nötig.

In [ ]:
# ─────────────────────────────────────────────────────
#  NUR DIESEN BLOCK ANPASSEN
IMG_TO_PREDICT = r'C:\Pfad\zu\deinem\bild.jpg'  # ← hier Pfad eintragen
TRUE_LAT = None   # optional: echte Koordinaten für Vergleich
TRUE_LON = None
# ─────────────────────────────────────────────────────

# Zellen-Mapping & Modell laden (falls Notebook neu gestartet)
if not Path(CFG['best_model']).exists():
    print('⚠️  Kein gespeichertes Modell gefunden – bitte zuerst Training ausführen.')
else:
    with open('checkpoints/cell_mapping.json') as f:
        mapping = json.load(f)
    idx_to_cell = {int(k): v for k, v in mapping['idx_to_cell'].items()}
    STEP = mapping['step']
    NUM_CLASSES = len(idx_to_cell)

    model_inf = GeoModel(CFG['backbone'], NUM_CLASSES, CFG['dropout']).to(DEVICE)
    ckpt_inf  = torch.load(CFG['best_model'], map_location=DEVICE)
    model_inf.load_state_dict(ckpt_inf['model_state'])
    model = model_inf

    if Path(IMG_TO_PREDICT).exists():
        predict_image(IMG_TO_PREDICT, TRUE_LAT, TRUE_LON)
    else:
        print(f'Datei nicht gefunden: {IMG_TO_PREDICT}')
        print('Demo mit zufälligem Test-Bild:')
        sample = df_test.sample(1).iloc[0]
        predict_image(sample['img_path'], sample['lat'], sample['lon'])

---
## Zusammenfassung

| Schritt | Was passiert |
|---------|-------------|
| **Dataset** | Google Street View via kagglehub |
| **Gitter** | Welt in `grid_deg`×`grid_deg` Zellen eingeteilt |
| **Modell** | EfficientNet-B4 (pretrained) + Klassifikations-Kopf |
| **Training** | CrossEntropy + AdamW + Cosine LR |
| **Metrik** | Haversine-Distanz (km), GCD-Kurve | 
| **Output** | Land, Stadt, Lat/Lon + Karte |

**Verbesserungsmöglichkeiten:**
- Hierarchische Vorhersage (mehrere Gittergranularitäten)
- Test-Time Augmentation (TTA)
- Größeres Backbone (EfficientNet-B7, ViT)
- Google Reverse Geocoding API (präziser als Nominatim)
- Ensemble mehrerer Modelle